In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import minimize_scalar

In [2]:
df = pd.read_csv('demand_estimates.csv').set_index('SKU')
df

,Low_Price,Med_Price,High_Price,Low_Price_Demand,Med_Price_Demand,High_Price_Demand,Unit_cost
SKU,,,,,,,
NW033,1.39,2.25,2.99,100,40,5,1.1
NW037,2.25,2.97,3.99,200,130,25,1.5
J0008,44.97,74.99,94.99,400,300,130,40.0
J0010,198.85,249.95,295.99,600,300,50,120.0


In [3]:
product_list = df.index.tolist()

In [4]:
results = []

for x in product_list:
    
    # look up product code and isolate a product's row
    product = df.loc[x]
    
    # re-arrange input values for fitting the demand curve
    prices = product[
        ['Low_Price', 'Med_Price', 'High_Price']
    ].tolist()
    
    demands = product[
        ['Low_Price_Demand', 'Med_Price_Demand', 'High_Price_Demand']
    ].tolist()
    
    # fitting the demand curve
    a, b, c = np.polyfit(prices, demands, 2)
    
    def demand(price):
        return a * price**2 + b * price + c
    
    def profit(price):
        return demand(price) * (price - product['Unit_cost'])
    
    # optimize for profit
    result = minimize_scalar(
        lambda price:-profit(price),
        bounds=(product['Low_Price'], product['High_Price']),
        method='bounded'
    )
    
    results.append([x, result.x, -result.fun])
    
pricing = pd.DataFrame(results, columns=['SKU', 'Optimized Price', 'Maximized Profit'])

In [5]:
# formatting the result table
pricing_styled = pricing.style.hide(axis=0).format("${:,.2f}", subset=['Optimized Price', 'Maximized Profit'])
display(pricing_styled)

SKU,Optimized Price,Maximized Profit
NW033,$1.97,$49.81
NW037,$2.89,$191.75
J0008,$78.52,"$10,631.63"
J0010,$209.32,"$47,929.29"
